# Phase 5: Advanced Analyses

## Objective
Round out the project with the four "unique analyses" the Advanced Track expects:
1. **Climate Trend Analysis** — long-term patterns in temperature and humidity
2. **Air Quality ↔ Weather Correlation** — how weather conditions drive pollution
3. **Feature Importance** — which weather variables most influence temperature
4. **Spatial Visualization** — global map of temperature and air quality

## Why these analyses?
Forecasting is just one part of a strong data project. These analyses turn the dataset into **insights** — about climate, environment, and geography — that go beyond predictions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
import os
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

os.makedirs("../figures", exist_ok=True)
def save_fig(name):
    plt.savefig(f"../figures/{name}.png", dpi=120, bbox_inches='tight')

In [ ]:
df = pd.read_parquet("../data/weather_cleaned.parquet")
print(f"Shape: {df.shape}")

## 1. Climate Trend Analysis

Are global temperatures rising during our observation window? We'll compute monthly global averages and look for trend signals.

**Caveat**: 2 years is too short to draw climate change conclusions — but we can still observe seasonal patterns and year-over-year comparisons.

In [ ]:
df['year_month'] = df['last_updated'].dt.to_period('M').astype(str)
monthly_global = df.groupby('year_month').agg(
    avg_temp=('temperature_celsius', 'mean'),
    avg_humidity=('humidity', 'mean'),
    avg_pm25=('air_quality_PM2.5', 'mean')
).reset_index()

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

axes[0].plot(monthly_global['year_month'], monthly_global['avg_temp'],
             marker='o', color='#E63946', linewidth=2)
axes[0].set_title('Global Avg Temperature Over Time', fontweight='bold')
axes[0].set_ylabel('Temperature (\u00b0C)')
axes[0].grid(alpha=0.3)

axes[1].plot(monthly_global['year_month'], monthly_global['avg_humidity'],
             marker='o', color='#457B9D', linewidth=2)
axes[1].set_title('Global Avg Humidity Over Time', fontweight='bold')
axes[1].set_ylabel('Humidity (%)')
axes[1].grid(alpha=0.3)

axes[2].plot(monthly_global['year_month'], monthly_global['avg_pm25'],
             marker='o', color='#F4A261', linewidth=2)
axes[2].set_title('Global Avg PM2.5 Over Time', fontweight='bold')
axes[2].set_ylabel('PM2.5 (\u00b5g/m\u00b3)')
axes[2].grid(alpha=0.3)

plt.xticks(rotation=45)
axes[2].set_xlabel('Month')
plt.tight_layout()
save_fig('phase5_climate_trends')
plt.show()

In [ ]:
# Year-over-year temperature comparison by month
df['_year'] = df['last_updated'].dt.year
df['_month'] = df['last_updated'].dt.month

yoy = df.groupby(['_year', '_month'])['temperature_celsius'].mean().unstack(level=0)

plt.figure(figsize=(12, 6))
for year in yoy.columns:
    plt.plot(yoy.index, yoy[year], marker='o', linewidth=2, label=str(year))

plt.title('Year-over-Year Monthly Avg Temperature (Global)', fontsize=14, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Avg Temperature (\u00b0C)')
plt.xticks(range(1, 13))
plt.legend(title='Year', fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
save_fig('phase5_year_over_year')
plt.show()

## 2. Air Quality ↔ Weather Correlation

Which weather conditions drive pollution? We'll examine PM2.5 against:
- Wind speed (high wind → disperses pollution)
- Humidity (affects particulate suspension)
- Pressure (high pressure traps pollution)
- Temperature (affects chemistry and ground-level inversions)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
weather_vars = ['wind_kph', 'humidity', 'pressure_mb', 'temperature_celsius']

for ax, var in zip(axes.flat, weather_vars):
    # Sample for plot speed
    sample = df.sample(min(20000, len(df)), random_state=42)
    ax.scatter(sample[var], sample['air_quality_PM2.5'], s=2, alpha=0.2, color='#457B9D')

    # Show binned median (the trend line)
    bins = pd.cut(df[var], bins=20)
    binned = df.groupby(bins)['air_quality_PM2.5'].median()
    bin_centers = [interval.mid for interval in binned.index]
    ax.plot(bin_centers, binned.values, color='#E63946', linewidth=2.5, label='Median trend')

    corr = df[[var, 'air_quality_PM2.5']].corr().iloc[0, 1]
    ax.set_title(f'{var} vs PM2.5 (corr={corr:.2f})', fontweight='bold')
    ax.set_xlabel(var)
    ax.set_ylabel('PM2.5')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('How Weather Affects Air Quality (PM2.5)', fontsize=15, fontweight='bold', y=1.00)
plt.tight_layout()
save_fig('phase5_aqi_vs_weather')
plt.show()

In [ ]:
# Air quality by weather condition category
common_conditions = df['condition_text'].value_counts().head(10).index
cond_aq = df[df['condition_text'].isin(common_conditions)].groupby('condition_text').agg(
    median_pm25=('air_quality_PM2.5', 'median'),
    avg_humidity=('humidity', 'mean'),
    count=('air_quality_PM2.5', 'count')
).sort_values('median_pm25', ascending=True)

plt.figure(figsize=(11, 6))
cond_aq['median_pm25'].plot(kind='barh', color='#F4A261', edgecolor='white')
plt.title('Median PM2.5 by Weather Condition', fontsize=14, fontweight='bold')
plt.xlabel('Median PM2.5 (\u00b5g/m\u00b3)')
plt.ylabel('')
plt.grid(alpha=0.3, axis='x')
plt.tight_layout()
save_fig('phase5_aqi_by_condition')
plt.show()

print("\nMedian PM2.5 by condition:")
print(cond_aq.round(2))

## 3. Feature Importance — What Drives Temperature?

We'll train a Random Forest to predict temperature from all other weather variables, then extract feature importances. This tells us **which features matter most** for temperature prediction.

In [ ]:
feature_cols = [
    'humidity', 'cloud', 'wind_kph', 'pressure_mb', 'precip_mm',
    'visibility_km', 'uv_index', 'air_quality_PM2.5', 'air_quality_Ozone',
    'latitude', 'longitude', 'month', 'hour', 'day_of_year'
]

# Use a sample for speed
sample = df.sample(min(40000, len(df)), random_state=42)
X = sample[feature_cols]
y = sample['temperature_celsius']

rf = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(X, y)

importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=True)

plt.figure(figsize=(11, 7))
plt.barh(importance_df['feature'], importance_df['importance'], color='#06A77D', edgecolor='white')
plt.title('Feature Importance for Predicting Temperature', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.grid(alpha=0.3, axis='x')
plt.tight_layout()
save_fig('phase5_feature_importance')
plt.show()

print("\nTop 5 most important features:")
print(importance_df.tail(5)[::-1].to_string(index=False))

## 4. Spatial Visualization — Global Map

A scatter map of all 257 cities, colored by their average temperature (and a second view by air quality). Visually shows how climate varies geographically.

In [ ]:
city_summary = df.groupby('location_name').agg(
    latitude=('latitude', 'first'),
    longitude=('longitude', 'first'),
    avg_temp=('temperature_celsius', 'mean'),
    avg_pm25=('air_quality_PM2.5', 'mean'),
    country=('country', 'first')
).reset_index()

fig, axes = plt.subplots(2, 1, figsize=(15, 12))

# Temperature map
scatter1 = axes[0].scatter(
    city_summary['longitude'], city_summary['latitude'],
    c=city_summary['avg_temp'], cmap='RdYlBu_r', s=60,
    edgecolor='black', linewidth=0.5, alpha=0.85
)
axes[0].set_title('Global Average Temperature by Capital City',
                  fontsize=14, fontweight='bold')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].axhline(0, color='black', linestyle='--', alpha=0.3, label='Equator')
axes[0].set_xlim(-180, 180)
axes[0].set_ylim(-60, 80)
axes[0].grid(alpha=0.3)
axes[0].legend()
plt.colorbar(scatter1, ax=axes[0], label='Avg Temperature (\u00b0C)')

# Air quality map
scatter2 = axes[1].scatter(
    city_summary['longitude'], city_summary['latitude'],
    c=city_summary['avg_pm25'].clip(upper=80),  # cap for visibility
    cmap='YlOrRd', s=60, edgecolor='black', linewidth=0.5, alpha=0.85
)
axes[1].set_title('Global Average Air Quality (PM2.5) by Capital City',
                  fontsize=14, fontweight='bold')
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')
axes[1].axhline(0, color='black', linestyle='--', alpha=0.3, label='Equator')
axes[1].set_xlim(-180, 180)
axes[1].set_ylim(-60, 80)
axes[1].grid(alpha=0.3)
axes[1].legend()
plt.colorbar(scatter2, ax=axes[1], label='Avg PM2.5 (\u00b5g/m\u00b3, capped at 80)')

plt.tight_layout()
save_fig('phase5_spatial_maps')
plt.show()

In [ ]:
# Top/bottom highlights
print("Top 5 hottest capitals:")
print(city_summary.nlargest(5, 'avg_temp')[['location_name', 'country', 'avg_temp']].to_string(index=False))

print("\nTop 5 coldest capitals:")
print(city_summary.nsmallest(5, 'avg_temp')[['location_name', 'country', 'avg_temp']].to_string(index=False))

print("\nTop 5 most polluted capitals (avg PM2.5):")
print(city_summary.nlargest(5, 'avg_pm25')[['location_name', 'country', 'avg_pm25']].to_string(index=False))

## ✅ Phase 5 Complete

**What we covered:**
1. **Climate trends** — monthly global averages and year-over-year comparison
2. **Air quality drivers** — how wind, humidity, pressure, and temperature affect PM2.5
3. **Feature importance** — which variables most influence temperature predictions
4. **Global spatial maps** — temperature and pollution by city across the world

**Saved figures:**
- `phase5_climate_trends.png`
- `phase5_year_over_year.png`
- `phase5_aqi_vs_weather.png`
- `phase5_aqi_by_condition.png`
- `phase5_feature_importance.png`
- `phase5_spatial_maps.png`

**Next:** Phase 6 — README, demo video, submission.